# 03 · Baseline model

A classic TF-IDF + logistic regression classifier in Spark ML, trained on the silver data. It sets the score AraBERT has to beat: if a bag-of-words model gets close, the fine-tuning isn't adding much.

| | |
|---|---|
| **Input** | `data/silver/hard_reviews/` (train / val / test) |
| **Output** | `data/export/{train,val,test}/`, a slim copy (`review_id`, `review_clean`, `label`) for the AraBERT notebook |

## Load the splits

The splits were fixed in silver, so they are just filtered here. `train` is cached because it is reused for every model in the tuning loop.

In [1]:
import time
from config import get_spark, SILVER_PATH
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = get_spark("baseline")

silver = spark.read.parquet(SILVER_PATH)
train = silver.filter("split = 'train'").cache()
val   = silver.filter("split = 'val'")
test  = silver.filter("split = 'test'")
print("train:", train.count(), " val:", val.count(), " test:", test.count())

train: 82821  val: 10217  test: 10217


## Pipeline

1. **RegexTokenizer**: splits on whitespace and punctuation, including the Arabic comma `،`, semicolon `؛` and question mark `؟`.
2. **CountVectorizer**: word counts over a vocabulary of up to 50,000 words, ignoring words that appear in fewer than `minDF` reviews.
3. **IDF**: down-weights words that appear in most reviews.
4. **LogisticRegression**: the classifier, with L2 regularisation (`regParam`).

In [2]:
tokenizer = RegexTokenizer(inputCol="review_clean", outputCol="words",
                           pattern=r"[\s\p{Punct}،؛؟]+", gaps=True)

cv = CountVectorizer(inputCol="words", outputCol="tf", vocabSize=50000, minDF=3)

idf = IDF(inputCol="tf", outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50, regParam=0.01)

pipeline = Pipeline(stages=[tokenizer, cv, idf, lr])

## Tune on the validation set

Small grid search over `minDF` (vocabulary cut-off) and `regParam` (regularisation strength), 9 models in total, each scored on **val**. The test set is not touched here, so it stays an honest final check.

Lower values of both work best: **minDF=20, regParam=0.1 → 85.15% val accuracy.**

In [3]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

best_acc, best_model, best_params = 0, None, None

for min_df in [20, 50, 100]:
    for reg in [0.1, 0.3, 1.0]:
        cv.setMinDF(min_df)
        lr.setRegParam(reg)
        m = pipeline.fit(train)
        acc = evaluator.evaluate(m.transform(val))
        print(f"minDF={min_df:>3}  regParam={reg:<4}  VAL acc={acc:.4f}")

        if acc > best_acc:
            best_acc, best_model, best_params = acc, m, (min_df, reg)

model = best_model
print(f"\nBEST: minDF={best_params[0]}  regParam={best_params[1]}  VAL acc={best_acc:.4f}")

minDF= 20  regParam=0.1   VAL acc=0.8515
minDF= 20  regParam=0.3   VAL acc=0.8493
minDF= 20  regParam=1.0   VAL acc=0.8395
minDF= 50  regParam=0.1   VAL acc=0.8499
minDF= 50  regParam=0.3   VAL acc=0.8455
minDF= 50  regParam=1.0   VAL acc=0.8349
minDF=100  regParam=0.1   VAL acc=0.8450
minDF=100  regParam=0.3   VAL acc=0.8400
minDF=100  regParam=1.0   VAL acc=0.8263

BEST: minDF=20  regParam=0.1  VAL acc=0.8515


## Evaluate the best model

Accuracy and F1 on val, plus a confusion matrix. Accuracy and F1 are nearly equal because the classes are balanced. The model makes more false positives (865 negative reviews predicted positive) than false negatives (652).

In [4]:
pred_val = model.transform(val)

def score(pred, name):
    acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
    f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
    print(f"{name}: accuracy={acc:.4f}  F1={f1:.4f}")

score(pred_val, "VAL")
    
pred_val.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

VAL: accuracy=0.8515  F1=0.8515
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 4304|
|    0|       1.0|  865|
|    1|       0.0|  652|
|    1|       1.0| 4396|
+-----+----------+-----+



## What the model learned

The words with the most negative and most positive weights. They make sense: `سيء` / `سيئة` (bad), `قديم` (old), `حشرات` (insects) on the negative side; `ممتاز` (excellent), `رائع` (wonderful), `الهدوء` (quiet) on the positive side.

The same word often shows up in several spellings (`سيء`, `سيئ`, `سئ`, `سيى`). A bag-of-words model treats each as a separate word, which is one of the limits AraBERT should handle better.

In [5]:
vocab = model.stages[1].vocabulary
coefs = model.stages[-1].coefficients.toArray()

ranked = sorted(zip(coefs, vocab))
print("MOST NEGATIVE:", [w for _, w in ranked[:20]])
print("MOST POSITIVE:", [w for _, w in ranked[-20:]])

MOST NEGATIVE: ['غير', 'سيئة', 'سيء', 'سيئه', 'سئ', 'لايستحق', 'سيئ', 'قديم', 'فقط', 'سوء', 'لا', 'متدنيه', 'ولاشي', 'سيى', 'حشرات', 'لن', 'السكارى', 'فاشل', 'البعوض', 'المستعان']
MOST POSITIVE: ['حسن', 'ممتعه', 'استمتعت', 'وهدوء', 'جميله', 'اجرائات', 'رائعه', 'وحسن', 'ومريح', 'وسرعة', 'راقي', 'رائعة', 'روعه', 'ممتازه', 'والهدوء', 'ممتازة', 'الهدوء', 'جميل', 'رائع', 'ممتاز']


## Export for AraBERT

One Parquet file per split with only the columns the transformer needs. `coalesce(1)` writes a single file per split so it is easy to load with pandas or Hugging Face `datasets`.

In [6]:
from config import ROOT
EXPORT = str(ROOT / "data" / "export")

for split in ["train", "val", "test"]:
    (silver.filter(F.col("split") == split)
        .select("review_id", "review_clean", "label")
        .coalesce(1)
        .write.mode("overwrite").parquet(f"{EXPORT}/{split}"))